# Data Integration Notebook
## JSON API Ingestion and XML Parsing

Demonstrates working with semi-structured data formats (JSON, XML),
parsing and loading them into SQL Server.

### Contents
1. SQL Server connection setup
2. JSON ingestion — REST Countries API
3. XML parsing and loading

In [ ]:
# conncection setup for SQL Server using pyodbc
import pyodbc
import pandas as pd
import requests
import json
from sqlalchemy import create_engine
import urllib

# pyodbc connection for pd.read_sql
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=RetailDW;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# SQLAlchemy engine for df.to_sql
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=RetailDW;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
connection_url = f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(connection_string)}"
engine = create_engine(connection_url)

print("Connected successfully")

In [ ]:
# ── JSON INGESTION ──────────────────────────────────────────────
# Pull country metadata from REST Countries public API
# Demonstrates: API endpoint call, JSON response parsing, 
# field extraction from nested JSON, loading to SQL Server

# Get countries from our retail database
countries_df = pd.read_sql("""
    SELECT DISTINCT Country FROM retail_sales
    WHERE Country != 'Unspecified'
    ORDER BY Country
""", conn)

# Map retail dataset country names to ISO alpha-2 codes
# Needed because API uses ISO codes, not free-text names
country_map = {
    'EIRE': 'IE', 'Czech Republic': 'CZ', 'Channel Islands': 'JE',
    'European Community': None, 'RSA': 'ZA', 'USA': 'US',
    'United Kingdom': 'GB', 'Netherlands': 'NL', 'Germany': 'DE',
    'France': 'FR', 'Spain': 'ES', 'Belgium': 'BE', 'Switzerland': 'CH',
    'Portugal': 'PT', 'Italy': 'IT', 'Finland': 'FI', 'Sweden': 'SE',
    'Norway': 'NO', 'Denmark': 'DK', 'Austria': 'AT', 'Greece': 'GR',
    'Cyprus': 'CY', 'Poland': 'PL', 'Malta': 'MT', 'Lithuania': 'LT',
    'Israel': 'IL', 'Iceland': 'IS', 'Canada': 'CA', 'Brazil': 'BR',
    'Australia': 'AU', 'Japan': 'JP', 'Bahrain': 'BH', 'Singapore': 'SG',
    'Lebanon': 'LB', 'United Arab Emirates': 'AE', 'Saudi Arabia': 'SA',
    'Hong Kong': 'HK'
}

country_data = []

for country in countries_df['Country'].tolist():
    try:
        code = country_map.get(country)
        if code is None:
            continue
        url = f"https://restcountries.com/v3.1/alpha/{code}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()[0]
            country_data.append({
                'Country': country,
                'Population': data.get('population', None),
                'Region': data.get('region', None),
                'Subregion': data.get('subregion', None),
                'Area_km2': data.get('area', None)
            })
    except:
        pass

df_enriched = pd.DataFrame(country_data)
print(f"Successfully retrieved: {len(df_enriched)} countries")
df_enriched.head()

In [ ]:
# load enriched country data back to SQL Server

df_enriched.to_sql(
    name='country_enriched',
    con=engine,
    if_exists='replace',
    index=False
)

print(f"Loaded {len(df_enriched)} rows into country_enriched table")